In [16]:
start_date_str = '2022-03-20 00:00:00 UTC'
end_date_str = '2023-03-19 23:59:59 UTC'
company = 'AAPL'

In [17]:
# Use a pipeline as a high-level helper
import os
#from huggingface_hub import InferenceClient
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
from datasets import load_dataset

In [18]:
#LOCAL COMPUTE
#from huggingface_hub import whoami, login

#Replace 'your_token' with your actual Hugging Face token
#HF_TOKEN =

# Pass your token directly
#login(token=HF_TOKEN)

#Verify login
#user = whoami()
#print(f"Logged in as {user['name']}")



In [19]:
# 1. Load in streaming mode to avoid the 'Generating' hang/crash
ds = load_dataset("sabareesh88/FNSPID_nasdaq")

# 2. Check the column names and data types (Features)
# Since it's streaming, we access the 'train' split metadata
print("Dataset Features:")
print(ds['train'].features)

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/48 [00:00<?, ?it/s]

Dataset Features:
{'Unnamed: 0': Value('string'), 'Date': Value('string'), 'Article_title': Value('string'), 'Stock_symbol': Value('string'), 'Url': Value('string'), 'Publisher': Value('string'), 'Author': Value('string'), 'Article': Value('string'), 'Lsa_summary': Value('string'), 'Luhn_summary': Value('string'), 'Textrank_summary': Value('string'), 'Lexrank_summary': Value('string')}


In [20]:
columns_to_drop = ['Luhn_summary', 'Textrank_summary', 'Lexrank_summary', 'Article']
# Much faster than .map()
ds_filtered = ds.remove_columns(columns_to_drop)

In [21]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# 1. Load the model and tokenizer locally
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
# Check if CUDA is available, otherwise use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def Sentiment_score(texts):
    all_results = []
    labels = ["Positive", "Negative", "Neutral"]

    for text_item in texts:
        # Tokenize and move to device
        inputs = tokenizer(
            text_item,
            truncation=True,
            max_length=510,
            add_special_tokens=True, # Good practice for classification
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)

        sentiment_scores = {label: predictions[0][j].item() for j, label in enumerate(labels)}
        all_results.append({
            "text": text_item,
            "sentiment_scores": sentiment_scores
        })
    return all_results

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
def filter_news_articles(dataset, start_date_str, end_date_str, company_symbol=None):
    from datetime import datetime
    """
    Filters a Hugging Face dataset using multiple CPU cores.
    """
    # 1. Pre-parse the filter bounds once (outside the loop)
    start_date = datetime.strptime(start_date_str, '%Y-%m-%d %H:%M:%S UTC')
    end_date = datetime.strptime(end_date_str, '%Y-%m-%d %H:%M:%S UTC')

    # Normalize symbol once
    target_symbol = company_symbol.upper() if company_symbol else None

    def filter_logic(example):
        # Handle missing or null dates
        date_str = example.get('Date')
        if not date_str:
            return False

        try:
            # Parse row date
            article_date = datetime.strptime(date_str, '%Y-%m-%d %H:%M:%S UTC')
            date_match = start_date <= article_date <= end_date

            # If date doesn't match, don't bother checking the symbol
            if not date_match:
                return False

            # If no symbol filter is requested, return True (date matched)
            if not target_symbol:
                return True

            # Symbol check
            stock_val = example.get('Stock_symbol')
            if stock_val:
                return target_symbol in stock_val.upper()

            return False

        except ValueError:
            return False

    # 2. Determine number of CPU cores to use
    # Using os.cpu_count() - 1 leaves one core free for your OS/background tasks
    cores = max(1, (os.cpu_count() or 1) - 1)

    print(f"Filtering dataset using {cores} cores...")

    # 3. Apply filter with num_proc
    return dataset.filter(filter_logic, num_proc=cores)

In [23]:
from datetime import datetime

print(f"Filtering news articles from {start_date_str} to {end_date_str}...")
filtered_results = filter_news_articles(ds_filtered, start_date_str, end_date_str, company_symbol=company)

print(f"Filter complete. Remaining rows: {len(filtered_results)}")

Filtering news articles from 2022-03-20 00:00:00 UTC to 2023-03-19 23:59:59 UTC...
Filtering dataset using 15 cores...
Filter complete. Remaining rows: 1


In [24]:
print(f"\n--- Exploring the first 3 rows of the date-filtered dataset ({start_date_str} to {end_date_str}) ---")
# Access the 'train' split and take the first 3 elements for enumeration
for i, row in enumerate(filtered_results['train'].take(3)):
    print(f"\nRow {i+1}:")
    for key, value in row.items():
        display_val = str(value)[:100] + "..." if len(str(value)) > 100 else value
        print(f"  {key}: {display_val}")


--- Exploring the first 3 rows of the date-filtered dataset (2022-03-20 00:00:00 UTC to 2023-03-19 23:59:59 UTC) ---

Row 1:
  Unnamed: 0: 16723.0
  Date: 2023-03-19 00:00:00 UTC
  Article_title: Buffett's Berkshire Hathaway speeds up stock buybacks
  Stock_symbol: AAPL
  Url: https://www.nasdaq.com/articles/buffetts-berkshire-hathaway-speeds-up-stock-buybacks
  Publisher: 
  Author: 
  Lsa_summary: The Omaha, Nebraska-based conglomerate owns dozens of businesses including Geico car insurance and t...

Row 2:
  Unnamed: 0: 16724.0
  Date: 2023-03-19 00:00:00 UTC
  Article_title: Why Apple, Warner Bros. Discovery, and AMD Are No-Brainer Buys Right Now
  Stock_symbol: AAPL
  Url: https://www.nasdaq.com/articles/why-apple-warner-bros.-discovery-and-amd-are-no-brainer-buys-right-n...
  Publisher: 
  Author: 
  Lsa_summary: Apple (NASDAQ: AAPL), Warner Bros. However, its stock has jumped 55% year to date, with investors fe...

Row 3:
  Unnamed: 0: 16725.0
  Date: 2023-03-19 00:00:00 UTC
  

In [25]:
#tokenizer = AutoTokenizer.from_pretrained(model_name)

#def tokenize_function(examples):
#    # This handles batches of strings at once
#   return tokenizer(examples["text"], padding="max_length", truncation=True)

# batched=True makes this significantly faster
#tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [26]:
# Extract summaries for sentiment analysis
article_summaries = [article['Lsa_summary'] for article in filtered_results['train']]

print(f"\nApplying sentiment analysis to {len(article_summaries)} article summaries...")
# Apply sentiment scoring
sentiment_results = Sentiment_score(article_summaries)

print("\nFirst 3 sentiment analysis results:")
for i, result in enumerate(sentiment_results[:3]):
    print(f"Result {i+1}:")
    print(f"  Text: {result['text'][:100]}...")
    print(f"  Scores: {result['sentiment_scores']}")


Applying sentiment analysis to 4167 article summaries...

First 3 sentiment analysis results:
Result 1:
  Text: The Omaha, Nebraska-based conglomerate owns dozens of businesses including Geico car insurance and t...
  Scores: {'Positive': 0.020260974764823914, 'Negative': 0.8535490036010742, 'Neutral': 0.12619005143642426}
Result 2:
  Text: Apple (NASDAQ: AAPL), Warner Bros. However, its stock has jumped 55% year to date, with investors fe...
  Scores: {'Positive': 0.01594286784529686, 'Negative': 0.9644933938980103, 'Neutral': 0.019563743844628334}
Result 3:
  Text: AT&T (NYSE: T) and Apple (NASDAQ: AAPL) are both considered stable blue-chip tech stocks to hold dur...
  Scores: {'Positive': 0.07089429348707199, 'Negative': 0.8699292540550232, 'Neutral': 0.05917638912796974}


In [27]:
def get_map_func(sentiment_data):
    def add_sentiment_scores_to_example(example, idx):
        if idx < len(sentiment_data):
            scores = sentiment_data[idx]['sentiment_scores']
            example['positive_score'] = scores.get('Positive', 0.0)
            example['negative_score'] = scores.get('Negative', 0.0)
            example['neutral_score'] = scores.get('Neutral', 0.0)
        else:
            example['positive_score'] = 0.0
            example['negative_score'] = 0.0
            example['neutral_score'] = 0.0
        return example
    return add_sentiment_scores_to_example

# Determine number of CPU cores to use for multiprocessing
cores = max(1, (os.cpu_count() or 1) - 1)

print(f"Adding sentiment scores to the dataset using {cores} cores...")

# Use it like this:
filtered_results['train'] = filtered_results['train'].map(
    get_map_func(sentiment_results), # We "inject" the results here
    with_indices=True,
    num_proc=cores
)

print("Sentiment scores added. Displaying the first entry with new scores:")
print(filtered_results['train'][0])

Adding sentiment scores to the dataset using 15 cores...
Sentiment scores added. Displaying the first entry with new scores:
{'Unnamed: 0': '16723.0', 'Date': '2023-03-19 00:00:00 UTC', 'Article_title': "Buffett's Berkshire Hathaway speeds up stock buybacks", 'Stock_symbol': 'AAPL', 'Url': 'https://www.nasdaq.com/articles/buffetts-berkshire-hathaway-speeds-up-stock-buybacks', 'Publisher': '', 'Author': '', 'Lsa_summary': 'The Omaha, Nebraska-based conglomerate owns dozens of businesses including Geico car insurance and the BNSF railroad, and stocks such as Apple Inc AAPL.O and Bank of America Corp BAC.N. They suggest that Buffett and fellow billionaire Vice Chairman Charlie Munger, who handle major capital allocation decisions, still view Berkshire\'s stock as undervalued, and repurchases as a prudent use of the company\'s cash. In his Feb. 25 annual letter to shareholders, Buffett defended buybacks, calling someone who views all repurchases as harmful "an economic illiterate or a silv

In [28]:
print("Filtered Results Dataset Features (including sentiment scores):")
print(filtered_results['train'].features)

Filtered Results Dataset Features (including sentiment scores):
{'Unnamed: 0': Value('string'), 'Date': Value('string'), 'Article_title': Value('string'), 'Stock_symbol': Value('string'), 'Url': Value('string'), 'Publisher': Value('string'), 'Author': Value('string'), 'Lsa_summary': Value('string'), 'positive_score': Value('float64'), 'negative_score': Value('float64'), 'neutral_score': Value('float64')}


In [29]:
df_filtered_results = filtered_results['train'].to_pandas()

df_filtered_results['Date'] = pd.to_datetime(df_filtered_results['Date'])
df_filtered_results['day'] = df_filtered_results['Date'].dt.date

daily_sentiment = df_filtered_results.groupby('day').agg(
    avg_positive_score=('positive_score', 'mean'),
    avg_negative_score=('negative_score', 'mean'),
    avg_neutral_score=('neutral_score', 'mean')
).reset_index()

print("Daily Aggregated Sentiment Scores (first 5 rows):")
display(daily_sentiment.head())

Daily Aggregated Sentiment Scores (first 5 rows):


,day,avg_positive_score,avg_negative_score,avg_neutral_score
0,2022-06-03,0.042049,0.729936,0.228015
1,2022-06-04,0.060833,0.064788,0.874379
2,2022-06-05,0.543850,0.015138,0.441012
3,2022-06-06,0.330284,0.333007,0.336709
4,2022-06-07,0.218690,0.249324,0.531986


In [30]:
# 2. Validation Phase
print("Running data validation...")

# Extract unique dates from both dataframes
source_dates = set(df_filtered_results['day'].unique())
aggregated_dates = set(daily_sentiment['day'].unique())

# Check: Are there any dates in the source that are missing in the aggregation?
missing_dates = source_dates - aggregated_dates

if not missing_dates:
    print("✅ Validation Passed: All dates accounted for.")
    
    # 3. Saving Phase
    output_filename = 'daily_sentiment.csv'
    daily_sentiment.to_csv(output_filename, index=False)
    print(f"Dataset successfully saved to {output_filename}")
    
    print("\nDaily Aggregated Sentiment Scores (first 5 rows):")
    display(daily_sentiment.head())
else:
    # Handle the error
    error_msg = f"❌ Validation Failed: {len(missing_dates)} dates are missing from the aggregation."
    print(error_msg)
    print("Missing dates include:", sorted(list(missing_dates))[:5]) # Show first 5 missing
    # We do not save the file if validation fails

Running data validation...
✅ Validation Passed: All dates accounted for.
Dataset successfully saved to daily_sentiment.csv

Daily Aggregated Sentiment Scores (first 5 rows):


,day,avg_positive_score,avg_negative_score,avg_neutral_score
0,2022-06-03,0.042049,0.729936,0.228015
1,2022-06-04,0.060833,0.064788,0.874379
2,2022-06-05,0.543850,0.015138,0.441012
3,2022-06-06,0.330284,0.333007,0.336709
4,2022-06-07,0.218690,0.249324,0.531986
